<a href="https://colab.research.google.com/github/shouvikcirca/LLMs/blob/deepeval/Copy_of_Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install torch
! pip install loralib
! pip install transformers
! pip install -U bitsandbytes
!pip install sentence-transformers # for embedding models
! pip install -U deepeval
! pip install python-dotenv

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig

# peft_model_id = "meta-llama/Llama-3.2-1B"
# config = PeftConfig.from_pretrained(peft_model_id)
# model = AutoModelForCausalLM.from_pretrained(peft_model_id, return_dict=True, load_in_8bit=True, device_map='auto')
# tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

# Load the Lora model
# model = PeftModel.from_pretrained(model, peft_model_id)

quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", quantization_config=quantization_config, device_map="auto")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
model.to('cuda')

RAG

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive', force_remount = True)

In [ ]:
excel_path = "drive/MyDrive/ccares_qna_dataset.xlsx"
query_df = pd.read_excel(excel_path)

In [ ]:
from sentence_transformers import SentenceTransformer, util
embedding_model = SentenceTransformer("all-mpnet-base-v2", device="cpu")

In [ ]:
# Extract the "Query" column
queries = query_df["Query"].tolist()

In [ ]:
# Create a DataFrame to hold query and metadata
query_data = pd.DataFrame({
    "query": queries,
    "chunk_char_count": [len(query) for query in queries],
    "chunk_word_count": [len(query.split()) for query in queries],
    "chunk_token_count": [len(query) // 4 for query in queries]  # Approx 4 chars per token
})

In [ ]:
from tqdm.auto import tqdm
# Create embeddings for all queries
print("[INFO] Generating embeddings for queries...")
query_data["embedding"] = list(tqdm(embedding_model.encode(queries, batch_size=32, show_progress_bar=True)))


[INFO] Generating embeddings for queries...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/229 [00:00<?, ?it/s]

In [ ]:
import torch
from sentence_transformers import util

# Semantic Search Function
def semantic_search(query, query_data, embedding_model, top_k):
    """
    Retrieve the top-k most relevant questions and return them with scores.
    """
    # Embed the input query
    query_embedding = embedding_model.encode(query, convert_to_tensor=True)

    # Extract stored embeddings
    stored_embeddings = torch.tensor(query_data["embedding"].tolist(), dtype=torch.float32)

    # Compute dot product scores
    scores = util.dot_score(query_embedding, stored_embeddings)[0]

    # Get the top-k results
    top_results = torch.topk(scores, k=top_k)

    # Store the top results
    retrieved_items = []
    print(f"Query: {query}\n")
    for score, idx in zip(top_results.values, top_results.indices):
        # Convert idx tensor to an integer
        idx = idx.item()
        score_value = score.item()

        # Only include results where the score is 0.50 or greater
        if score_value >= 0.50:
            result_query = query_data.iloc[idx]["query"]
            retrieved_items.append({"query": result_query, "score": score_value})

    return retrieved_items

# Example
input_query = "What is the purpose of the Coal Mines Provident Fund?"
retrieved_results = semantic_search(input_query, query_data, embedding_model, top_k=10)
print("\n[INFO] Retrieved Results:", retrieved_results)

Query: What is the purpose of the Coal Mines Provident Fund?


[INFO] Retrieved Results: [{'query': 'What is the purpose of the Coal Mines Provident Fund and Miscellaneous Provisions Act, 1948, in this context?', 'score': 0.8599842190742493}, {'query': 'What legal provision facilitates the application of the Coal Mines Provident Fund Scheme?', 'score': 0.8304412364959717}, {'query': 'Under which act is the Coal Mines Provident Fund Scheme governed?', 'score': 0.8144053220748901}, {'query': 'To which regions does the Coal Mines Provident Fund Scheme apply?', 'score': 0.793662428855896}, {'query': 'Which act is cited in the passage as governing the provident fund scheme for coal mines?', 'score': 0.7883508205413818}, {'query': "Which Indian provinces are mentioned as part of the Coal Mines Provident Fund Scheme's applicability?", 'score': 0.7729427218437195}, {'query': 'What is the Act number of the Coal Mines Provident Fund and Miscellaneous Provisions', 'score': 0.7707929611206055}, {'

<ipython-input-11-d03a1b4c8074>:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  stored_embeddings = torch.tensor(query_data["embedding"].tolist(), dtype=torch.float32)


In [ ]:
# Map Top-K Results with Corresponding Answers
def map_queries_to_answers(retrieved_results, query_df):
    """
    Maps retrieved queries to their corresponding answers from query_df.
    """
    mapped_results = []
    for item in retrieved_results:
        query_text = item["query"]
        score = item["score"]

        # Find the corresponding section in query_df
        section = query_df.loc[query_df["Query"] == query_text, "Section"].values

        # Find the corresponding answer in query_df
        answer = query_df.loc[query_df["Query"] == query_text, "Answer"].values
        if len(answer) > 0:
            answer = answer[0]  # Extract the first matching answer
        else:
            answer = "Answer not found."  # Handle cases where no match is found

        # Append the mapped query and answer
        mapped_results.append({"section": section, "query": query_text, "answer": answer, "score": score})

    return mapped_results



In [ ]:
mapped_results = map_queries_to_answers(retrieved_results, query_df)
# mapped_results
# Display the mapped results
for item in mapped_results:
    print(f"Query: {item['query']}\nAnswer: {item['answer']}\nScore: {item['score']:.4f}\nSection: {item['section']}\n")


Query: What is the purpose of the Coal Mines Provident Fund and Miscellaneous Provisions Act, 1948, in this context?
Answer: It provides the legal framework for the application of the Coal Mines Provident Fund Scheme to the mentioned regions.
Score: 0.8600
Section: ['Paragraph 1, Subparagraph 2']

Query: What legal provision facilitates the application of the Coal Mines Provident Fund Scheme?
Answer: The scheme is applied under Sub-section (1) of Section 92 of the Government of India Act, 1935.
Score: 0.8304
Section: ['Paragraph 1, Subparagraph 2']

Query: Under which act is the Coal Mines Provident Fund Scheme governed?
Answer: The scheme is governed by the Coal Mines Provident Fund and Miscellaneous Provisions Act, 1948.
Score: 0.8144
Section: ['Paragraph 1, Subparagraph 2']

Query: To which regions does the Coal Mines Provident Fund Scheme apply?
Answer: The scheme applies to all coal mines in West Bengal, Bihar, Maharashtra, the Central Provinces and Berar, Nagaland, and Odisha, in

In [ ]:
def prompt_formatter_with_mapped_results(query: str, context_items: list[dict]) -> str:
    """
    Formats the query with relevant context items for the Llama-based model.
    """
    # Construct context using the queries and answers
    context = "\n".join([item['section'][0]+'\n'+item["query"]+'\n'+item['answer'] for item in context_items])

    # Simplify instructions and emphasize single-query focus
    base_prompt = f"""Use the following context items to answer the query:
{context}

If the user query is not explicitly answered in the context, respond with "I don't know."
Only provide the answer to the user query and include references to the relevant sections. Do not provide answers to any follow-up questions.

User Query: {query}
Answer:"""

    return base_prompt


In [ ]:
input_query = "subsection (1) of Section 3-C in the definition of the Commissioner"
print(f"Query: {input_query}")
prompt = prompt_formatter_with_mapped_results(query=input_query,
                          context_items=mapped_results)

Query: subsection (1) of Section 3-C in the definition of the Commissioner


In [ ]:
print(prompt)

Use the following context items to answer the query:
Paragraph 1, Subparagraph 2
What is the purpose of the Coal Mines Provident Fund and Miscellaneous Provisions Act, 1948, in this context?
It provides the legal framework for the application of the Coal Mines Provident Fund Scheme to the mentioned regions.
Paragraph 1, Subparagraph 2
What legal provision facilitates the application of the Coal Mines Provident Fund Scheme?
The scheme is applied under Sub-section (1) of Section 92 of the Government of India Act, 1935.
Paragraph 1, Subparagraph 2
Under which act is the Coal Mines Provident Fund Scheme governed?
The scheme is governed by the Coal Mines Provident Fund and Miscellaneous Provisions Act, 1948.
Paragraph 1, Subparagraph 2
To which regions does the Coal Mines Provident Fund Scheme apply?
The scheme applies to all coal mines in West Bengal, Bihar, Maharashtra, the Central Provinces and Berar, Nagaland, and Odisha, including those in partially excluded areas in the provinces of W

Inference

In [ ]:

# Tokenize the prompt
input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate response with stricter settings
outputs = model.generate(
    **input_ids,
    temperature=0.3,  # More deterministic
    max_new_tokens=50,  # Restrict length to avoid additional responses
    top_p=0.9,  # Nucleus sampling
    do_sample=False  # Ensure deterministic generation
)

# Decode the response
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

# Extract only the model's answer
generated_answer = output_text[len(prompt):].strip().split("\n")[0]  # Take only the first answer
print(f"Generated Answer:\n{generated_answer}")


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Generated Answer:
I don't know.  The user query is not explicitly answered in the context. The context only mentions Sub-section (1) of Section 92 of the Government of India Act, 1935, and does not provide information about Section 3


In [ ]:
generated_answer

"I don't know.  The user query is not explicitly answered in the context. The context only mentions Sub-section (1) of Section 92 of the Government of India Act, 1935, and does not provide information about Section 3"

Evaluation

In [ ]:
from deepeval import evaluate
from deepeval.metrics import ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase

In [ ]:
from dotenv import load_dotenv
import os
_= load_dotenv('drive/MyDrive/.env')
openai_api_key = os.environ['OPENAI_API_KEY']

In [ ]:
actual_output = generated_answer

In [ ]:
retrieval_context = [i['section'][0]+'\n'+i['query']+'\n'+i['answer'] for i in mapped_results]

In [ ]:
metric = ContextualRelevancyMetric(
    threshold=0.7,
    model='gpt-3.5-turbo',
    include_reason=True
)

In [ ]:
test_case = LLMTestCase(
    input=input_query,
    actual_output=actual_output,
    retrieval_context=retrieval_context
)

In [ ]:
metric.measure(test_case)
print(metric.score)
# print(metric.reason)

Output()

Event loop is already running. Applying nest_asyncio patch to allow async execution...

ERROR:root:OpenAI rate limit exceeded. Retrying: 1 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 1 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 1 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 1 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 1 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 1 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 1 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 1 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 1 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 1 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 2 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 2 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 2 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 2 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 2 time(s)...
ERROR:root:OpenAI rate limit exceeded. Retrying: 2 time(s)...
ERROR:ro

KeyboardInterrupt: 